In [1]:
import os
import csv

folder_path = "./CIRT-122014-102025"  # randam select one folder from the Fannie Mae dataset 
target_loan_ids = ["45422765", "45476383", "45422752"]  # if more data want to be included, add more id here

output_file = "loan_records.csv"

# 1. Create buckets for each target loan ID
records = {loan_id: [] for loan_id in target_loan_ids}

# 2. Scan all CSV files and collect matching lines
for file in os.listdir(folder_path):
    if not file.endswith(".csv"):
        continue

    full_path = os.path.join(folder_path, file)

    with open(full_path, "r", encoding="utf-8") as f:
        for line in f:
            for loan_id in target_loan_ids:
                if loan_id in line:
                    records[loan_id].append(line.strip())
                    break

# 3. Write results in the specified loan ID order
with open(output_file, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.writer(out_f)
    writer.writerow(["raw_line"])

    for loan_id in target_loan_ids:
        for row in records[loan_id]:
            writer.writerow([row])

print("Done! All matching records have been written in loan ID order:", output_file)

Done! All matching records have been written in loan ID order: loan_records.csv


In [ ]:
# Define three cases/types, check: https://www.overleaf.com/read/qpxhymhmwbzn#0942da
# Record-level classification rules
BAD_ZB_CODES = {
    "03", "06", "09",
    "12", "13", "15", "16", "17",
    "96", "97", "98", "99",
}

DQ_INDEX = 39        # Delinquency status field index
ZB_CODE_INDEX = 43   # Zero balance code field index


def normalize_dq(raw_dq: str) -> str:
    """
    Normalize delinquency status value.

    Rules:
    - Treat R, empty string, NA, or 0 as current status 00
    - Otherwise keep the original value
    """
    dq = raw_dq.strip()
    if dq in {"R", "", "NA"}:
        return "00"
    if dq == "0":
        return "00"
    return dq


def classify_row_by_text_line(line: str):
    """
    Input: one full Fannie Mae pipe-separated text line
    Output: (label, dq_raw, dq_norm, zb_code)
    """
    parts = line.rstrip("\n").split("|")

    # Not enough fields to parse required indices
    if len(parts) <= max(DQ_INDEX, ZB_CODE_INDEX):
        return ("Unknown", None, None, None)

    dq_raw = parts[DQ_INDEX].strip()
    zb_code = parts[ZB_CODE_INDEX].strip()
    dq_norm = normalize_dq(dq_raw)

    # Case 1: Bad loan if zero balance code indicates default or termination
    if zb_code in BAD_ZB_CODES:
        return ("Bad", dq_raw, dq_norm, zb_code)

    try:
        dq_int = int(dq_norm)
    except ValueError:
        dq_int = 99

    # Case 2: Bad loan if delinquency is 3 months or more
    if dq_int >= 3:
        return ("Bad", dq_raw, dq_norm, zb_code)

    # Case 3: Average loan for 1 to 2 months delinquent
    if dq_norm in {"01", "02"}:
        return ("Average", dq_raw, dq_norm, zb_code)

    # Case 4: Otherwise treat as Good
    return ("Good", dq_raw, dq_norm, zb_code)


# Loan-level classification
def classify_loan_from_lines(lines):
    """
    Input:
        A list of raw text lines belonging to the same loan

    Output:
        loan_label, per_record_rows

    per_record_rows is a list of dicts.
    Each dict contains:
        period, label, dq_raw, dq_norm, zb, raw_line
    """
    has_bad = False
    has_avg = False
    rows = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        parts = line.split("|")
        period = parts[2] if len(parts) > 2 else ""

        label, dq_raw, dq_norm, zb = classify_row_by_text_line(line)

        rows.append(
            {
                "period": period,
                "label": label,
                "dq_raw": dq_raw,
                "dq_norm": dq_norm,
                "zb": zb,
                "raw_line": line,
            }
        )

        if label == "Bad":
            has_bad = True
        elif label == "Average":
            has_avg = True

    # Loan-level aggregation rule
    if has_bad:
        loan_label = "Bad"
    elif has_avg:
        loan_label = "Average"
    else:
        loan_label = "Good"

    return loan_label, rows

In [3]:
# Processing each loan (record-level first, then loan-level aggregation)
import csv

input_file = "loan_records.csv"
output_labeled = "loan_labeled.csv"
raw_lines = []

# 1. Read all raw_line entries from the input CSV
with open(input_file, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader, None)  # Skip header row

    # Each row contains one raw_line in the first column
    for row in reader:
        if not row:
            continue

        raw_line = row[0]
        if raw_line.strip():
            raw_lines.append(raw_line)

# 2. Perform loan-level aggregation and classification
loan_label, records = classify_loan_from_lines(raw_lines)


# 3. Write record-level results together with the loan-level label
with open(output_labeled, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.writer(out_f)
    writer.writerow(
        [
            "period",
            "label",
            "dq_raw",
            "dq_norm",
            "zb",
            "raw_line",
            "loan_label",
        ]
    )

    for r in records:
        writer.writerow(
            [
                r["period"],
                r["label"],
                r["dq_raw"],
                r["dq_norm"],
                r["zb"],
                r["raw_line"],
                loan_label,
            ]
        )

print("Processing complete. Results written to:", output_labeled)

Processing complete. Results written to: loan_labeled.csv
